# Lesson 10: Distributed sources

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

May 20th, 2026


In [ ]:
#!pip3 install mne-connectivity

In [ ]:
import matplotlib.pyplot as plt
import mne
from mne.datasets import sample, fetch_fsaverage
from mne.beamformer import make_lcmv, apply_lcmv
import numpy as np
import os.path as op
from mne.minimum_norm import make_inverse_operator, apply_inverse
from mne_connectivity.viz import plot_connectivity_circle
from mne.viz import circular_layout

## Data import

In [ ]:
data_path = sample.data_path()
subjects_dir = str(data_path) + '/subjects'
raw_fname = str(data_path) + '/MEG/sample/sample_audvis_filt-0-40_raw.fif'

raw = mne.io.read_raw_fif(raw_fname)
raw.info['bads'] = ['MEG 2443']  
event_id = 1  
tmin, tmax = -0.2, 0.5
events = mne.find_events(raw)
raw.pick(['meg', 'eog']) 
epochs = mne.Epochs(raw, events, event_id, tmin, tmax,
                    baseline=(None, 0), preload=True,
                    reject=dict(grad=4000e-13, mag=4e-12, eog=150e-6))
evoked = epochs.average().crop(0.05, 0.15)
evoked.plot_joint()

## Forward model

In [ ]:
fwd_fname = str(data_path) + '/MEG/sample/sample_audvis-meg-vol-7-fwd.fif'
forward = mne.read_forward_solution(fwd_fname)

## Covariance matrix

In [ ]:
data_cov = mne.compute_covariance(epochs, tmin=0.01, tmax=0.25,
                                  method='empirical')
noise_cov = mne.compute_covariance(epochs, tmin=tmin, tmax=0,
                                   method='empirical')
data_cov.plot(epochs.info)
noise_cov.plot(epochs.info)

In [ ]:
fig = epochs.average().plot_joint()

In [ ]:
fig = epochs.average().plot_white(noise_cov)

In [ ]:
fig = epochs.average().plot_white(data_cov)

## Spatial filter

In [ ]:
#Scalar
filters = make_lcmv(evoked.info, forward, data_cov, reg=0.05,
                    noise_cov=noise_cov, pick_ori='max-power',
                    weight_norm='unit-noise-gain', rank=None)
#Vector
filters_vec = make_lcmv(evoked.info, forward, data_cov, reg=0.05,
                        noise_cov=noise_cov, pick_ori='vector',
                        weight_norm='unit-noise-gain', rank=None)
src = forward['src']

In [ ]:
stc = apply_lcmv(evoked, filters)
stc_vec = apply_lcmv(evoked, filters_vec)

## Results of beamforming

In [ ]:
lims = [0.3, 0.45, 0.6]
kwargs = dict(src=src, subject='sample', subjects_dir=subjects_dir,
              initial_time=0.087, verbose=True)

In [ ]:
stc

In [ ]:
fig=stc.plot(mode='stat_map', clim=dict(kind='percent', pos_lims=lims), **kwargs)

In [ ]:
fig=stc.plot(mode='glass_brain', clim=dict(kind='value', lims=lims), **kwargs)

In [ ]:
fig=stc_vec.plot(mode='stat_map', clim=dict(kind='value', pos_lims=lims), **kwargs)

In [ ]:
fig=stc_vec.plot(mode='glass_brain', clim=dict(kind='value', lims=lims), **kwargs)

In [ ]:
peak_vox, _ = stc_vec.get_peak(tmin=0.08, tmax=0.1, vert_as_index=True)

ori_labels = ['x', 'y', 'z']
fig, ax = plt.subplots(1)
for ori, label in zip(stc_vec.data[peak_vox, :, :], ori_labels):
    ax.plot(stc_vec.times, ori, label='%s component' % label)
ax.legend(loc='lower right')
ax.set(title='Activity per orientation in the peak voxel', xlabel='Time (s)',
       ylabel='Amplitude (a. u.)')
mne.viz.utils.plt_show()
del stc_vec

## Play with covariance

In [ ]:
data_cov.data

In [ ]:
data_cov.data.shape

In [ ]:
for i in np.arange(0,305,5):
    for j in np.arange(0,305,5):
        if i!=j:
            data_cov.data[i,j]=0.9*np.sqrt(data_cov.data[i,i]*data_cov.data[j,j])
data_cov.plot(epochs.info)
        

In [ ]:
filters = make_lcmv(evoked.info, forward, data_cov, reg=0.05,
                    noise_cov=noise_cov, pick_ori='max-power',
                    weight_norm='unit-noise-gain', rank=None)
stc = apply_lcmv(evoked, filters)
stc_vec = apply_lcmv(evoked, filters_vec)

lims = [0.3, 0.45, 0.6]
kwargs = dict(src=src, subject='sample', subjects_dir=subjects_dir,
              initial_time=0.087, verbose=True)
fig=stc.plot(mode='stat_map', **kwargs)

In [ ]:
for i in range(305):
    for j in range(305):
        if i!=j:
            data_cov.data[i,j]=0*np.sqrt(data_cov.data[i,i]*data_cov.data[j,j])
data_cov.plot(epochs.info)
        

In [ ]:
filters = make_lcmv(evoked.info, forward, data_cov, reg=0.05,
                    noise_cov=noise_cov, pick_ori='max-power',
                    weight_norm='unit-noise-gain', rank=None)
stc = apply_lcmv(evoked, filters)
stc_vec = apply_lcmv(evoked, filters_vec)


fig=stc.plot(mode='stat_map', **kwargs)

In [ ]:
data_cov = mne.compute_covariance(epochs, tmin=0.01, tmax=0.25,
                                  method='empirical')
noise_cov = mne.compute_covariance(epochs, tmin=tmin, tmax=0,
                                   method='empirical')
for i in range(305):
    for j in range(305):
        if i!=j:
            noise_cov.data[i,j]=0*np.sqrt(noise_cov.data[i,i]*noise_cov.data[j,j])
data_cov.plot(epochs.info)
noise_cov.plot(epochs.info)

In [ ]:
filters = make_lcmv(evoked.info, forward, data_cov, reg=0.05,
                    noise_cov=noise_cov, pick_ori='max-power',
                    weight_norm='unit-noise-gain', rank=None)
stc = apply_lcmv(evoked, filters)
stc_vec = apply_lcmv(evoked, filters_vec)

fig=stc.plot(mode='stat_map', **kwargs)

## Minimum norm estimate

In [ ]:
data_path = sample.data_path()
subjects_dir = str(data_path) + '/subjects'

# Read data
fname_evoked = str(data_path) + '/MEG/sample/sample_audvis-ave.fif'
evoked = mne.read_evokeds(fname_evoked, condition='Left Auditory',
                          baseline=(None, 0))
fname_fwd = str(data_path) + '/MEG/sample/sample_audvis-meg-eeg-oct-6-fwd.fif'
fname_cov = str(data_path) + '/MEG/sample/sample_audvis-cov.fif'
fwd = mne.read_forward_solution(fname_fwd)
src = fwd['src']
cov = mne.read_cov(fname_cov)

## Regularization

In [ ]:
snr = 0.5
lambda2 = 1.0 / snr ** 2
kwargs = dict(initial_time=0.08, hemi='both', subjects_dir=subjects_dir,
              size=(600, 600))

In [ ]:
inv = make_inverse_operator(evoked.info, fwd, cov, loose=0., depth=0.8,
                            verbose=True)
stc = stc = abs(apply_inverse(evoked, inv, lambda2, 'MNE', verbose=True))
brain = stc.plot(figure=5, **kwargs)
brain.add_text(0.1, 0.9, 'MNE', 'title', font_size=14)

In [ ]:
snr = 10
lambda2 = 1.0 / snr ** 2
kwargs = dict(initial_time=0.08, hemi='both', subjects_dir=subjects_dir,
              size=(600, 600))

In [ ]:
inv = make_inverse_operator(evoked.info, fwd, cov, loose=0., depth=0.8,
                            verbose=True)
stc = stc = abs(apply_inverse(evoked, inv, lambda2, 'MNE', verbose=True))
brain = stc.plot(figure=5, **kwargs)
brain.add_text(0.1, 0.9, 'MNE', 'title', font_size=14)

## Impact of normalization

In [ ]:
snr = 10
lambda2 = 1.0 / snr ** 2
kwargs = dict(initial_time=0.08, hemi='both', subjects_dir=subjects_dir,
              size=(600, 600))

In [ ]:
inv = make_inverse_operator(evoked.info, fwd, cov, loose=0., depth=0.8,
                            verbose=True)
stc = abs(apply_inverse(evoked, inv, lambda2, 'MNE', verbose=True))
brain = stc.plot(figure=5, **kwargs)
brain.add_text(0.1, 0.9, 'MNE', 'title', font_size=14)

In [ ]:
inv = make_inverse_operator(evoked.info, fwd, cov, loose=0., depth=0.8,
                            verbose=True)
stc = abs(apply_inverse(evoked, inv, lambda2, 'dSPM', verbose=True))
brain = stc.plot(figure=5, **kwargs)
brain.add_text(0.1, 0.9, 'MNE', 'title', font_size=14)

## Using data covariance for inverse modeling

In [ ]:
epochs.average().plot_joint()

In [ ]:
noise_cov = mne.make_ad_hoc_cov(epochs.info)
base_cov = mne.compute_covariance(
    epochs, tmin=-0.2, tmax=0, method='shrunk', verbose=True)
data_cov = mne.compute_covariance(
    epochs, tmin=0.07, tmax=0.13, method='shrunk', verbose=True)


In [ ]:

fig_base_cov = mne.viz.plot_cov(base_cov, epochs.info, show_svd=False)
fig_data_cov = mne.viz.plot_cov(data_cov, epochs.info, show_svd=False)

In [ ]:
fig = base_cov.plot_topomap(evoked.info)
fig = data_cov.plot_topomap(evoked.info)

In [ ]:
inv = make_inverse_operator(epochs.info, fwd, cov, loose=0., depth=0.8,
                            verbose=True)

In [ ]:
stc_data = mne.minimum_norm.apply_inverse_cov(data_cov, evoked.info, inv,
                             nave=len(epochs), method='dSPM', verbose=True)
stc_base = mne.minimum_norm.apply_inverse_cov(base_cov, evoked.info, inv,
                             nave=len(epochs), method='dSPM', verbose=True)

In [ ]:
brain = stc_base.plot(figure=5, **kwargs)

In [ ]:
brain = stc_data.plot(figure=5, **kwargs)

In [ ]:
stc_data /= stc_base
brain = stc_data.plot(figure=5, **kwargs)

## Source leakage

In [ ]:
from mne.minimum_norm import (read_inverse_operator,
                              make_inverse_resolution_matrix,
                              get_point_spread)

In [ ]:
fname_fwd = str(data_path) + '/MEG/sample/sample_audvis-meg-eeg-oct-6-fwd.fif'
fname_inv = str(data_path) + '/MEG/sample/sample_audvis-meg-oct-6-meg-fixed-inv.fif'
forward = mne.read_forward_solution(fname_fwd)
mne.convert_forward_solution(
    forward, surf_ori=True, force_fixed=True, copy=False)
inv = read_inverse_operator(fname_inv)

In [ ]:
labels = mne.read_labels_from_annot('sample', parc='aparc',
                                    subjects_dir=subjects_dir)


In [ ]:
labels

In [ ]:
label_colors = [label.color for label in labels]
# First, we reorder the labels based on their location in the left hemi
label_names = [label.name for label in labels]
lh_labels = [name for name in label_names if name.endswith('lh')]

# Get the y-location of the label
label_ypos = list()
for name in lh_labels:
    idx = label_names.index(name)
    ypos = np.mean(labels[idx].pos[:, 1])
    label_ypos.append(ypos)

# Reorder the labels based on their location
lh_labels = [label for (yp, label) in sorted(zip(label_ypos, lh_labels))]

# For the right hemi
rh_labels = [label[:-2] + 'rh' for label in lh_labels]

In [ ]:
rm_mne = make_inverse_resolution_matrix(forward, inv,
                                        method='MNE', lambda2=1. / 3.**2)

In [ ]:
plt.scatter(rm_mne[0,:], forward['sol']['data'][0,:])

In [ ]:
rm_mne.shape

In [ ]:


n_comp = 5
stcs_psf_mne, pca_vars_mne = get_point_spread(
    rm_mne, src, labels, mode="pca", n_comp=n_comp, norm=None, return_pca_vars=True
)
n_verts = rm_mne.shape[0]
del rm_mne

In [ ]:
stcs_psf_mne

In [ ]:
len(stcs_psf_mne)

In [ ]:
idx=0
labels[idx]

In [ ]:
stcs_psf_mne[idx].plot(figure=5, subject='sample', subjects_dir= subjects_dir, hemi='both')

In [ ]:
with np.printoptions(precision=1):
    for [name, var] in zip(label_names, pca_vars_mne):
        print(f'{name}: {var.sum():.1f}% {var}')

In [ ]:
n_labels=len(labels)
# get PSFs from Source Estimate objects into matrix
psfs_mat = np.zeros([n_labels, n_verts])
# Leakage matrix for MNE, get first principal component per label
for [i, s] in enumerate(stcs_psf_mne):
    psfs_mat[i, :] = s.data[:, 0]
# Compute label-to-label leakage as Pearson correlation of PSFs
# Sign of correlation is arbitrary, so take absolute values
leakage_mne = np.abs(np.corrcoef(psfs_mat))

# Save the plot order and create a circular layout
node_order = lh_labels[::-1] + rh_labels  # mirror label order across hemis
node_angles = circular_layout(label_names, node_order, start_pos=90,
                              group_boundaries=[0, len(label_names) / 2])
# Plot the graph using node colors from the FreeSurfer parcellation. We only
# show the 200 strongest connections.

plot_connectivity_circle(leakage_mne, label_names, n_lines=200,
                         node_angles=node_angles, node_colors=label_colors,
                         title='MNE Leakage')

## Other inverse operators?

In [ ]:
inv = make_inverse_operator(evoked.info, fwd, noise_cov, loose=0., depth=0.8,
                            verbose=True)

In [ ]:
snr = 3.0
lambda2 = 1.0 / snr ** 2
kwargs = dict(initial_time=0.08, hemi='both', subjects_dir=subjects_dir,
              size=(600, 600))
stc = abs(apply_inverse(evoked, inv, lambda2, 'MNE', verbose=True))
brain = stc.plot(figure=1, **kwargs)
brain.add_text(0.1, 0.9, 'MNE', 'title', font_size=14)

In [ ]:
snr = 3.0
lambda2 = 1.0 / snr ** 2
kwargs = dict(initial_time=0.08, hemi='both', subjects_dir=subjects_dir,
              size=(600, 600))
stc = abs(apply_inverse(evoked, inv, lambda2, 'dSPM', verbose=True))
brain = stc.plot(figure=1, **kwargs)
brain.add_text(0.1, 0.9, 'dSPM', 'title', font_size=14)

In [ ]:
snr = 3.0
lambda2 = 1.0 / snr ** 2
kwargs = dict(initial_time=0.08, hemi='both', subjects_dir=subjects_dir,
              size=(600, 600))
stc = abs(apply_inverse(evoked, inv, lambda2, 'eLORETA', verbose=True))
brain = stc.plot(figure=1, **kwargs)
brain.add_text(0.1, 0.9, 'eLORETA', 'title', font_size=14)